<a href="https://colab.research.google.com/github/jaeho0726/Sentiment-Analysis/blob/main/Sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **1. Setup and Dependencies**

In [ ]:
!set -x \
&& pip install konlpy \
&& curl -s https://raw.githubusercontent.com/konlpy/master/scripts/mecab.sh | bash

## **2. Import Libraries and Configuration**


In [ ]:
import re

import urllib.request
import numpy as np
import pandas as pd

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from torch.utils.data import Dataset
import torch.nn.functional as F

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
sns.set_style("white")

plt.rcParams['font.family'] = 'NanumBarunGothic'
plt.rcParams['axes.unicode_minus'] = False

from collections import Counter

from scipy.stats import pearsonr

## **3. Naver Movie Review Data: Loading and Initial Cleaning**

In [ ]:
# Accessing Naver movie review data
train_file = urllib.request.urlopen("https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt")
test_file = urllib.request.urlopen("https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt")

train_data = pd.read_table(train_file)
test_data = pd.read_table(test_file)

train_data[:10]

In [ ]:
# Finding numbers of data in train data
print(train_data['document'].nunique())
print(train_data['label'].nunique())

train_data.drop_duplicates(subset=['document'], inplace = True)

In [ ]:
# Finding missing values in train data
print(train_data.isnull().sum())

train_data = train_data.dropna(how='any')

In [ ]:
# Getting rid of characters except Korean characters and blank spaces
train_data['document'] = train_data['document'].str.replace("[^ㄱ-ㅎㅏ-ㅣ가-힣 ]", "", regex=True)

train_data[:10]

In [ ]:
train_data['document'] = train_data['document'].replace('', np.nan)
print(len(train_data))
print(train_data.isnull().sum())

In [ ]:
train_data = train_data.dropna(how='any')
print(len(train_data))

In [ ]:
# Loading the KLUE-BERT Tokenizer and Model
tokenizer = AutoTokenizer.from_pretrained("klue/bert-base")

class NSMCDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.encodings = tokenizer(texts, truncation=True, padding='max_length', max_length=max_len)
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# Prepare the data objects
train_dataset = NSMCDataset(train_data['document'].tolist(), train_data['label'].tolist(), tokenizer)
test_dataset = NSMCDataset(test_data['document'].tolist(), test_data['label'].tolist(), tokenizer)


## **4. KLUE-BERT Model Preparation and Training**

In [ ]:
# Loading pre-trained KLUE-BERT model for classification
model = AutoModelForSequenceClassification.from_pretrained("klue/bert-base", num_labels=2)

# Setting up training arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    logging_steps=100,
    fp16=True if torch.cuda.is_available() else False
)

# Initialize the Trainer and start training
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

trainer.train()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
def bert_predict(text):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        prob_positive = F.softmax(outputs.logits, dim=-1)[0][1].item()

        return (prob_positive - 0.5) * 200

## **5. Daily Journal Data: Loading and Sentiment Analysis**

In [ ]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default

creds, _ = default()

gc = gspread.Client(auth=creds)

spreadsheet = gc.open_by_url('https://docs.google.com/spreadsheets/d/1DrDCLitpXZyJajcWYcMNcJCZU4Xa0aE_gj4sfXzl6x4/edit?gid=0#gid=0')

worksheet = spreadsheet.worksheet('data')

In [ ]:
raw = worksheet.get_all_records()

headers = raw[0]
rows = raw[1:]

df = pd.DataFrame(rows, columns=headers)

# Shortening Column Names
df = df.rename(columns={
    "Work Intensity (0 - Easy / 10 - Intense)": "Work Intensity",
    "Overall Mood (0 - Poor / 10 - Great)": "Overall Mood",
    "Name": "Date"
})

# Fixing Data Types
df["Hours of Work"] = pd.to_numeric(df["Hours of Work"], errors="coerce").fillna(0)
df["Work Intensity"] = pd.to_numeric(df["Work Intensity"], errors="coerce").fillna(0)
df["Overall Mood"] = pd.to_numeric(df["Overall Mood"], errors="coerce").fillna(0)
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(by='Date')
df['Day_of_Week'] = df['Date'].dt.day_name()


df

In [ ]:
df['Sentiment Score'] = df['Reflection'].apply(bert_predict)

df

In [ ]:
def categorize_sentiment(score):
  if score > 0:
    return '긍정'
  elif score < 0:
    return '부정'
  else:
    return '중립'

df['Sentiment'] = df['Sentiment Score'].apply(categorize_sentiment)

df

## **6. Visualization**

In [ ]:
on_duty = df[df["Today's Status"] == "On Duty"].copy()

pos_df = df[df['Sentiment'] == '긍정'].copy()
neg_df = df[df['Sentiment'] == '부정'].copy()

In [ ]:
def extract_korean_words(text_series):
    words = []
    for text in text_series:
        if isinstance(text, str):
            found = re.findall(r'[가-힣]{2,}', text)
            words.extend(found)
    return Counter(words)

# Common filler words to exclude
filler = {'오늘', '정말', '것이', '것도', '것을', '것은', '것같', '같다', '같은',
          '하지만', '그래도', '이번', '생각', '때문', '느낌', '그리고', '하면서',
          '있는', '있었', '없는', '없었', '했던', '했다', '했는데', '했고',
          '이라', '에서', '에게', '으로', '에도', '했을', '이었', '이다',
          '한다', '한것', '해서', '하고', '하는', '이번에', '오늘도', '그래서'}

def top_words(counter, n=30):
    return [(w, c) for w, c in counter.most_common(200)
            if w not in filler][:n]

print("Extracting words...")
pos_words = top_words(extract_korean_words(pos_df['Reflection']))
neg_words = top_words(extract_korean_words(neg_df['Reflection']))

In [ ]:
# High mood but negative sentiment
false_neg = df[(df['Overall Mood'] >= 7) & (df['Sentiment Score'] < 0)].copy()
# Low mood but positive sentiment
false_pos = df[(df['Overall Mood'] <= 4) & (df['Sentiment Score'] > 0)].copy()

df['Discrepancy Type'] = 'Aligned'
df.loc[false_neg.index, 'Discrepancy Type'] = 'High Mood / Negative Text'
df.loc[false_pos.index, 'Discrepancy Type'] = 'Low Mood / Positive Text'

In [ ]:
fig = plt.figure(figsize=(20, 28))
gs  = gridspec.GridSpec(nrows=4, ncols=2, figure=fig, hspace=0.55, wspace=0.35)

# Positive Word Bubble Chart
ax1 = fig.add_subplot(gs[0, 0])
ax1.set_xlim(0, 1)
ax1.set_ylim(0, 1)
ax1.axis('off')
ax1.set_title('Top Words — Positive Entries (긍정)', fontsize=13, pad=10)

if pos_words:
    max_freq = pos_words[0][1]
    np.random.seed(42)
    placed = []
    for word, freq in pos_words:
        size   = 300 + (freq / max_freq) * 2500
        radius = np.sqrt(size) / 250
        for _ in range(500):
            x, y = np.random.uniform(radius, 1 - radius), np.random.uniform(radius, 1 - radius)
            if all(np.sqrt((x-px)**2 + (y-py)**2) > radius + pr + 0.02
                   for px, py, pr in placed):
                placed.append((x, y, radius))
                ax1.scatter(x, y, s=size, color='#378ADD', alpha=0.6,
                            edgecolors='white', linewidths=0.8)
                ax1.text(x, y, word, ha='center', va='center',
                         fontsize=max(7, int(6 + (freq / max_freq) * 8)),
                         color='#042C53', fontweight='500')
                break

# Negative Word Bubble Chart
ax2 = fig.add_subplot(gs[0, 1])
ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)
ax2.axis('off')
ax2.set_title('Top Words — Negative Entries (부정)', fontsize=13, pad=10)

if neg_words:
    max_freq = neg_words[0][1]
    np.random.seed(7)
    placed = []
    for word, freq in neg_words:
        size   = 300 + (freq / max_freq) * 2500
        radius = np.sqrt(size) / 250
        for _ in range(500):
            x, y = np.random.uniform(radius, 1 - radius), np.random.uniform(radius, 1 - radius)
            if all(np.sqrt((x-px)**2 + (y-py)**2) > radius + pr + 0.02
                   for px, py, pr in placed):
                placed.append((x, y, radius))
                ax2.scatter(x, y, s=size, color='#E24B4A', alpha=0.6,
                            edgecolors='white', linewidths=0.8)
                ax2.text(x, y, word, ha='center', va='center',
                         fontsize=max(7, int(6 + (freq / max_freq) * 8)),
                         color='#501313', fontweight='500')
                break

# Mood vs Sentiment Discrepancy Scatter
ax3 = fig.add_subplot(gs[1, :])
color_map = {
    'Aligned':                 '#888780',
    'High Mood / Negative Text': '#E24B4A',
    'Low Mood / Positive Text':  '#378ADD'
}
for dtype, group in df.groupby('Discrepancy Type'):
    ax3.scatter(group['Overall Mood'], group['Sentiment Score'],
                label=f"{dtype} (n={len(group)})",
                color=color_map[dtype], alpha=0.8, s=80,
                edgecolors='white', linewidths=0.5)

ax3.axhline(0,  color='gray', linestyle='--', linewidth=1, alpha=0.6)  # neutral line
ax3.axvline(5,  color='gray', linestyle='--', linewidth=1, alpha=0.6)
ax3.set_title('Mood vs Sentiment Score — Discrepancy Analysis\n(KLUE-BERT: score range -100 to +100, 0 = neutral)',
              fontsize=13)
ax3.set_xlabel('Overall Mood (0–10)')
ax3.set_ylabel('Sentiment Score (-100 to +100)')
ax3.legend(fontsize=10)

ax3.text(0.3, 85,  'Low Mood\nPositive Text',  fontsize=8, color='#378ADD', alpha=0.8)
ax3.text(7.5, 85,  'High Mood\nPositive Text', fontsize=8, color='#888780', alpha=0.8)
ax3.text(0.3, -90, 'Low Mood\nNegative Text',  fontsize=8, color='#888780', alpha=0.8)
ax3.text(7.5, -90, 'High Mood\nNegative Text', fontsize=8, color='#E24B4A', alpha=0.8)

# Hours of Work vs Sentiment Score (On Duty Only)
ax4 = fig.add_subplot(gs[2, 0])
colors_duty = on_duty['Sentiment'].map({'긍정': '#378ADD', '부정': '#E24B4A', '중립': '#888780'})
ax4.scatter(on_duty['Hours of Work'], on_duty['Sentiment Score'],
            c=colors_duty, alpha=0.7, edgecolors='white', linewidths=0.5)
if len(on_duty) > 1:
    m, b = np.polyfit(on_duty['Hours of Work'], on_duty['Sentiment Score'], 1)
    x_line = np.linspace(on_duty['Hours of Work'].min(), on_duty['Hours of Work'].max(), 100)
    ax4.plot(x_line, m * x_line + b, color='gray', linestyle='--', linewidth=1.2)
ax4.axhline(0, color='gray', linestyle=':', linewidth=1, alpha=0.5)
ax4.set_title('Hours of Work vs Sentiment\n(On Duty Days Only)', fontsize=13)
ax4.set_xlabel('Hours of Work')
ax4.set_ylabel('Sentiment Score (-100 to +100)')

# Work Intensity vs Sentiment Score (On Duty Only)
ax5 = fig.add_subplot(gs[2, 1])
ax5.scatter(on_duty['Work Intensity'], on_duty['Sentiment Score'],
            c=colors_duty, alpha=0.7, edgecolors='white', linewidths=0.5)
if len(on_duty) > 1:
    m2, b2 = np.polyfit(on_duty['Work Intensity'], on_duty['Sentiment Score'], 1)
    x_line2 = np.linspace(on_duty['Work Intensity'].min(), on_duty['Work Intensity'].max(), 100)
    ax5.plot(x_line2, m2 * x_line2 + b2, color='gray', linestyle='--', linewidth=1.2)
ax5.axhline(0, color='gray', linestyle=':', linewidth=1, alpha=0.5)
ax5.set_title('Work Intensity vs Sentiment\n(On Duty Days Only)', fontsize=13)
ax5.set_xlabel('Work Intensity (0–10)')
ax5.set_ylabel('Sentiment Score (-100 to +100)')

In [ ]:
# Calculate Pearson correlation between Sentiment Score and Overall Mood
# Note: pearsonr returns the correlation coefficient and the p-value.
correlation_coefficient, p_value = pearsonr(df['Sentiment Score'], df['Overall Mood'])

print(f"Pearson Correlation between Sentiment Score and Overall Mood: {correlation_coefficient:.3f}")
print(f"P-value: {p_value:.3f}")

if correlation_coefficient > 0.7:
    print("There is a strong positive correlation, suggesting the model's sentiment largely aligns with your overall mood.")
elif correlation_coefficient > 0.3:
    print("There is a moderate positive correlation.")
elif correlation_coefficient < -0.7:
    print("There is a strong negative correlation, which might indicate a mismatch or inverse relationship.")
elif correlation_coefficient < -0.3:
    print("There is a moderate negative correlation.")
else:
    print("The correlation is weak, suggesting a limited linear relationship between the sentiment score and overall mood.")